# Plain Language Government Navigator - Gemma 4 E4B Fine-tuning

This notebook fine-tunes Google's Gemma 4 E4B-it model using Unsloth + QLoRA for the Plain Language Government Navigator.
The fine-tuned model is then exported to GGUF format for deployment via Ollama.

**Environment:** Kaggle notebook with T4 GPU (16 GB VRAM)  
**Dataset:** Navigator training data (chat-format JSONL with system/user/assistant turns)  
**Output:** Q4_K_M quantized GGUF model + Ollama Modelfile

## 1. Setup & Install

Install Unsloth and its dependencies. We use `--no-deps` for trl/peft/accelerate/bitsandbytes
to avoid pulling in conflicting transitive dependencies that Kaggle already provides.

In [ ]:
# Install Unsloth (Kaggle-compatible)
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes

## 2. Imports

Core libraries: Unsloth for optimized model loading and PEFT, TRL for supervised fine-tuning,
and HuggingFace datasets for data handling.

In [ ]:
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset
import json, torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## 3. Load Model

Load Gemma 4 E4B-it via Unsloth's optimized loader with 4-bit quantization (NF4).
This reduces VRAM usage from ~16 GB to ~4 GB, fitting comfortably on a T4.

We then attach LoRA adapters (rank 16) to all attention and MLP projection layers,
giving us a trainable parameter count of roughly 1-2% of the full model.

In [ ]:
MAX_SEQ_LENGTH = 2048

# Load base model with 4-bit quantization
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-4-E4B-it",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,  # auto-detect (bf16 on Ampere+, fp16 on Turing)
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

print(f"Trainable parameters: {model.print_trainable_parameters()}")

## 4. Load Training Data

Training data is a JSONL file where each line contains a `messages` array in OpenAI chat format:

```json
{"messages": [
    {"role": "system", "content": "You are a plain-language benefits navigator..."},
    {"role": "user", "content": "I lost my job and have two kids..."},
    {"role": "assistant", "content": "Here are programs you may be eligible for..."}
]}
```

Upload the dataset as a Kaggle dataset and reference it via the standard input path.

In [ ]:
DATASET_PATH = "/kaggle/input/mn-navigator-training/final.jsonl"

# Parse JSONL into list of dicts
records = []
with open(DATASET_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

dataset = Dataset.from_list(records)

print(f"Training examples: {len(dataset)}")
print(f"\nFirst example:")
print(json.dumps(dataset[0], indent=2))

## 5. Training Configuration

Key choices:
- **Effective batch size 4** (batch=1 x grad_accum=4) to fit in T4 VRAM
- **Cosine LR scheduler** with 10% warmup for stable convergence
- **bf16** for mixed precision (falls back to fp16 on older GPUs)
- **adamw_8bit** optimizer to reduce memory footprint
- **3 epochs** -- enough for a small domain-specific dataset without overfitting

In [ ]:
OUTPUT_DIR = "/kaggle/working/navigator-finetune"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=5,
    save_steps=50,
    save_total_limit=2,
    fp16=False,
    bf16=True,
    optim="adamw_8bit",
    seed=42,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=training_args,
    max_seq_length=MAX_SEQ_LENGTH,
)

print(f"Training config:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Optimizer: {training_args.optim}")
print(f"  Output: {OUTPUT_DIR}")

## 6. Train

Run the fine-tuning loop. On a Kaggle T4, expect roughly 10-20 minutes depending on
dataset size. Watch for the training loss to decrease steadily -- if it plateaus early,
the dataset may be too small or the learning rate too low.

In [ ]:
train_result = trainer.train()

# Print training stats
metrics = train_result.metrics
print(f"\nTraining complete!")
print(f"  Total steps: {metrics.get('total_flos', 'N/A')}")
print(f"  Training loss: {metrics.get('train_loss', 'N/A'):.4f}")
print(f"  Training runtime: {metrics.get('train_runtime', 0):.1f}s")
print(f"  Samples/second: {metrics.get('train_samples_per_second', 0):.2f}")

## 7. Save LoRA Adapters

Save the trained LoRA adapter weights separately. These are small (~50-100 MB)
and can be merged with the base model later for inference or GGUF export.

In [ ]:
FINAL_DIR = "/kaggle/working/navigator-finetune/final"

model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

print(f"LoRA adapters saved to: {FINAL_DIR}")

# Show saved files
import os
for f in sorted(os.listdir(FINAL_DIR)):
    size_mb = os.path.getsize(os.path.join(FINAL_DIR, f)) / 1024 / 1024
    print(f"  {f} ({size_mb:.1f} MB)")

## 8. Export to GGUF

Convert the fine-tuned model to GGUF format with Q4_K_M quantization.
This produces a single file (~2-3 GB) that Ollama can load directly.

Q4_K_M is a good balance of quality and size -- it uses 4-bit quantization with
medium-sized K-quant lookup tables for better accuracy than plain Q4_0.

In [ ]:
GGUF_DIR = "/kaggle/working/navigator-gguf"

model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method="q4_k_m",
)

print(f"GGUF model saved to: {GGUF_DIR}")

# Show the exported file
for f in sorted(os.listdir(GGUF_DIR)):
    size_mb = os.path.getsize(os.path.join(GGUF_DIR, f)) / 1024 / 1024
    print(f"  {f} ({size_mb:.1f} MB)")

## 9. Create Ollama Modelfile

Generate an Ollama Modelfile that packages the GGUF weights with the Navigator's
system prompt. After downloading the GGUF from Kaggle output, run:

```bash
ollama create navigator -f Modelfile
ollama run navigator
```

In [ ]:
# Find the GGUF file in the output directory
gguf_files = [f for f in os.listdir(GGUF_DIR) if f.endswith(".gguf")]
assert len(gguf_files) > 0, f"No GGUF files found in {GGUF_DIR}"
gguf_filename = gguf_files[0]

# Navigator system prompt (from src/navigator/prompts.py with standard/English defaults)
SYSTEM_PROMPT = """You are a plain-language government benefits navigator. Your job is to explain \
benefits eligibility results in clear, simple language.

TARGET READING LEVEL: standard
- simple: 5th grade reading level. Short sentences. No jargon. Define any \
technical terms. Use "you" and "your."
- standard: 8th grade reading level. Clear but can use some common terms.
- detailed: Full detail. Can use technical terms but define acronyms on first use.

LANGUAGE: Respond in English.

You are presenting results to a real person in a difficult situation. Be warm, \
clear, and actionable. Structure your response as:

1. A brief empathetic acknowledgment (1 sentence)
2. HIGH PRIORITY programs (most impactful/urgent)
3. ALSO CHECK programs (additional options)
4. DOCUMENTS TO GATHER (consolidated list across all programs)
5. WHERE TO APPLY (grouped by application portal with direct URLs)

For each program, include:
- Program name and what it provides (1 sentence)
- Why they may be eligible (1 sentence citing the rule)
- Estimated benefit amount if available
- Any important deadlines or notes

End with the standard disclaimer.

CRITICAL: Only include programs from the provided eligibility results. Do not \
add programs not in the results. Do not fabricate benefit amounts."""

modelfile_content = f"""FROM ./{gguf_filename}

PARAMETER temperature 1.0
PARAMETER top_p 0.95

SYSTEM \"\"\"
{SYSTEM_PROMPT}
\"\"\"
"""

modelfile_path = os.path.join(GGUF_DIR, "Modelfile")
with open(modelfile_path, "w") as f:
    f.write(modelfile_content)

print(f"Modelfile written to: {modelfile_path}")
print()
print("--- Modelfile contents ---")
print(modelfile_content)

## 10. Test Inference

Quick sanity check with the fine-tuned model (before GGUF conversion, using the
full-precision LoRA-merged model in memory). This confirms the fine-tuning shifted
the model's behavior toward our domain.

In [ ]:
# Switch to inference mode
FastLanguageModel.for_inference(model)

# Sample benefits question
test_messages = [
    {
        "role": "user",
        "content": (
            "I'm a single mom with two kids ages 3 and 7. I just lost my job "
            "last week and I live in Hennepin County. My income was about $32,000 "
            "a year. What help is available for me?"
        ),
    },
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=512,
    temperature=1.0,
    top_p=0.95,
    do_sample=True,
)

# Decode only the generated tokens (skip the input)
response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

print("Question:")
print(test_messages[0]["content"])
print()
print("Response:")
print(response)